In [1]:
import math
import pandas as pd
from statistics import NormalDist
import numpy as np

### Task 1 - Compare AOV using Ratio of Means to AOV using Mean of per-user ratios

In [2]:
experiment = pd.read_csv("../data/experiment.csv", parse_dates = ["exposure_date"])
experiment.head()

,user_id,variant,exposure_date,pre_gmv_28d,orders_14d,gmv_14d
0,14357,treatment,2026-09-04,1534.52,4,1441.69
1,3440,control,2026-09-04,1817.86,1,368.36
2,12154,control,2026-09-01,0.00,1,421.78
3,15030,treatment,2026-09-06,0.00,1,258.44
4,18550,treatment,2026-09-01,518.17,1,166.94


In [3]:
experiment["own_aov"] = experiment["gmv_14d"]/experiment["orders_14d"]
control_df = experiment[experiment["variant"] == "control"].copy()
treatment_df = experiment[experiment["variant"] == "treatment"].copy()

# AOV using ratio of means
control_rom = control_df["gmv_14d"].sum()/control_df["orders_14d"].sum()
treatment_rom = treatment_df["gmv_14d"].sum()/treatment_df["orders_14d"].sum()

# AOV using mean of per-user ratios
control_mor = control_df["own_aov"].mean()
treatment_mor = treatment_df["own_aov"].mean()

print(f"Control's Ratio of Mean: {control_rom:.4f} and Mean of Ratios: {control_mor:.4f}")
print(f"Treatments's Ratio of Mean: {treatment_rom:.4f} and Mean of Ratios: {treatment_mor:.4f}")

Control's Ratio of Mean: 315.9038 and Mean of Ratios: 318.0196
Treatments's Ratio of Mean: 333.1594 and Mean of Ratios: 335.2812


In [4]:
print(f"Users dropped from treatment: {treatment_df['own_aov'].isnull().sum()} out of {len(treatment_df)}")
print(f"Users dropped from control: {control_df['own_aov'].isnull().sum()} out of {len(control_df)}")

Users dropped from treatment: 2305 out of 6298
Users dropped from control: 2299 out of 5702


Q. How many users gets dropped in AOV using mean of per-user ratios? 

Ans. 4604 out of 12000 (38.37%) users were dropped due to no orders made. It is risky because the treatment that can change who orders, and filtering them leaves a different mix in each arm and the comparison stops being between randomized groups.

Q. Why the AOV using ratio of means is different from AOV using mean of per-user ratios?

Ans. AOV using mean of per-user ratios is just not ordinary AOV metrics but actually user-weighted AOV metrics. Observe the calculation, Ratio of Means is (GMV/users)/(Orders/users) which becomes AOV while Means of Ratios is first calculation of AOV of individual users and then its mean.

### Task 2 - Use `ratio_delta` function to get z-stat, p_value and 95% confidenc interval

In [5]:
import sys; sys.path.insert(0, "../src")
from ds_prep.stats_utils import ratio_delta

In [6]:
# Treatment Ratio Delta
num_t = treatment_df["gmv_14d"].to_list()
den_t = treatment_df["orders_14d"].to_list()

result_t = ratio_delta(num_t, den_t)
r_t = result_t["r"]
se_t = result_t["se"]

# Control Ratio Delta
num_c = control_df["gmv_14d"].to_list()
den_c = control_df["orders_14d"].to_list()

result_c = ratio_delta(num_c, den_c)
r_c = result_c["r"]
se_c = result_c["se"]

# z-stat
## diff
diff = r_t - r_c
## standard error
se = ((se_t**2) + (se_c)**2)**0.5
print(f"Standard Error: {se:.4f}")
## z-stat
z = diff / se
print(f"z: {z:.4f}")

# p-value
p_value = math.erfc(abs(z)/math.sqrt(2))
print(f"p-value: {p_value:.2e}")

# 95% Confidence Interval
z_critical = NormalDist().inv_cdf(0.975)
ci_low = diff - z_critical * se
ci_high = diff + z_critical * se
print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

Standard Error: 5.8205
z: 2.9646
p-value: 3.03e-03
95% Confidence Interval: [5.8475, 28.6635]


### Task 3 - Use `bootstrap` to get z-stat, p_value and 95% confidenc interval

In [7]:
from scipy.stats import bootstrap

In [8]:
def ratio(y, x, axis=-1):
    return y.sum(axis=axis) / x.sum(axis=axis)

res = {}

for arm, g in experiment.groupby("variant"):
    res[arm] = bootstrap(
        (g["gmv_14d"].to_numpy(), g["orders_14d"].to_numpy()),
        ratio,
        paired=True,          # resample users, keep (GMV, orders) together
        vectorized=True,      # ratio() handles a whole batch via axis
        n_resamples=2000,
        batch=200,            # 200 resamples at a time → no memory blow-up
        method="percentile",
        random_state=0,
    )

# Standard Error
se = ((res["treatment"].standard_error**2) + (res["control"].standard_error**2))**0.5
print(f"Standard Error: {se:.4f}")

# 95% COnfidence Interval
d = res["treatment"].bootstrap_distribution - res["control"].bootstrap_distribution
ci = np.percentile(d, [2.5, 97.5])
ci_low = ci[0]
ci_high = ci[1]
print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")


Standard Error: 5.7378
95% Confidence Interval: [6.2903, 28.5562]


Looking at SE 5.82 vs 5.74 and CIs [5.85, 28.66] vs [6.29, 28.56] we can say `ratio_delta` method and `bootstrap` closely agree with each other. So that means `ratio_delta` function is working properly with Taylor approximation for n = 6000. And since CIs also nearly coincide for both, sampling distribution is close to normal

### Task 4 - Use `orders.csv` and compute the overall AOV SE two ways: the naive order-level version, sd(order GMV)/√(total orders), and user-level delta SE

In [9]:
orders = pd.read_csv("../data/orders.csv", parse_dates = ["order_ts"])
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery


In [10]:
# Order-level version
total_gmv = orders["gmv"].sum()
gmv_sd = orders["gmv"].std()
total_orders = len(orders)

se_ol = gmv_sd/(total_orders**0.5)
r_ol = total_gmv/total_orders

# User-level version
user_orders = (
    orders.groupby(["user_id"], as_index = False)
    .agg(
        gmv = ("gmv", "sum"),
        orders = ("order_id", "nunique")
    )
)

num = user_orders["gmv"].to_list()
den = user_orders["orders"].to_list()
result = ratio_delta(num, den)

se_ul = result["se"]
r_ul = result["r"]

# Print to compare
print(f"Order-level Standard Error: {se_ol:.4f}, User-level Standard Error: {se_ul:.4f}")
print(f"Order-level Ratio: {r_ol:.4f}, User-level Ratio: {r_ul:.4f}")

Order-level Standard Error: 1.0479, User-level Standard Error: 1.0510
Order-level Ratio: 428.3197, User-level Ratio: 428.3197


Q. Is the gap big on this data? Explain why.

Ans. The gap between standard errors aren't big because data was generated through simulation script and it has no concept of shared per user component. In real world, user's orders are correlated so as per Design Effect the naive order-level SE understates the true variance by a factor of DEFF, and the gap grows with more orders per user (m) and stronger habits (ρ).